# RAG for IoT Privacy Policies & Product Descriptions

A ready-to-run notebook to build a small Retrieval-Augmented Generation (RAG) system over **privacy policies** and **product descriptions** for popular smart-home devices using **local Ollama models**.

**What you'll get:**
- A mini corpus built from privacy policy pages, product pages, and (optionally) manuals
- Chunking + embeddings with an open sentence model (`all-MiniLM-L6-v2`)
- A FAISS vector index for semantic retrieval
- A RAG pipeline that uses **Ollama** for local LLM generation (no API keys needed!)

**Prerequisites:**
1. **Install Ollama**: Download from https://ollama.ai or run `brew install ollama`
2. **Start Ollama**: Run `ollama serve` in terminal
3. **Pull a model**: Run `ollama pull llama3` (or your preferred model)

**How to use:**
1. Run the setup cells (installs + imports).
2. Check Ollama status using the management utilities.
3. Inspect or edit the starter device list (URLs included).
4. Build the dataset and index.
5. Ask questions like _"What data does Ring Doorbell collect?"_

**Supported models:** llama3, llama3.1, mistral, phi3, codellama, and others available in Ollama.

> ⚠️ Scraping ethics: Respect robots.txt and terms of service. This demo pulls a few public pages. For production, implement rate limiting, error handling, and caching.

In [ ]:
# %%capture
!pip -q install beautifulsoup4 requests pandas pypdf sentence-transformers faiss-cpu tiktoken openai duckduckgo-search

## Imports & Config

## Ollama Setup & Management
Quick utilities to check Ollama status, list models, and pull new models if needed.

In [10]:
import requests
import os

# Ollama Configuration for local LLM
class OllamaConfig:
    def __init__(self):
        self.ollama_url = os.getenv('OLLAMA_URL', 'http://localhost:11434')
        self.model_name = os.getenv('LLM_MODEL', 'llama3:latest')
        self.timeout = int(os.getenv('LLM_TIMEOUT', '30'))
        self.max_retries = int(os.getenv('LLM_MAX_RETRIES', '3'))
    
    def is_available(self) -> bool:
        """Check if Ollama is running and accessible"""
        try:
            response = requests.get(f"{self.ollama_url}/api/tags", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def get_available_models(self) -> List[str]:
        """Get list of available models from Ollama"""
        try:
            response = requests.get(f"{self.ollama_url}/api/tags", timeout=5)
            if response.status_code == 200:
                models = response.json().get('models', [])
                return [model['name'] for model in models]
        except:
            return []
        return []

# Initialize Ollama config
ollama_config = OllamaConfig()

def check_ollama_status():
    """Check Ollama status and available models"""
    if ollama_config.is_available():
        models = ollama_config.get_available_models()
        print(f"✅ Ollama is running at {ollama_config.ollama_url}")
        print(f"📋 Available models: {', '.join(models) if models else 'None'}")
        
        if ollama_config.model_name in models:
            print(f"🎯 Current model '{ollama_config.model_name}' is available")
        else:
            print(f"⚠️ Current model '{ollama_config.model_name}' not found")
            if models:
                print(f"💡 You can use: {models[0]}")
    else:
        print(f"❌ Ollama not available at {ollama_config.ollama_url}")
        print("💡 Start Ollama with: ollama serve")
        print(f"💡 Pull model with: ollama pull {ollama_config.model_name}")

def pull_ollama_model(model_name: str = None):
    """Pull a model in Ollama"""
    model_name = model_name or ollama_config.model_name
    try:
        url = f"{ollama_config.ollama_url}/api/pull"
        payload = {"name": model_name}
        print(f"⬇️ Pulling model '{model_name}'... (this may take a while)")
        
        response = requests.post(url, json=payload, timeout=300)
        if response.status_code == 200:
            print(f"✅ Model '{model_name}' pulled successfully")
            return True
        else:
            print(f"❌ Failed to pull model: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error pulling model: {e}")
        return False

def test_ollama_model(model_name: str = None):
    """Test if a model is working"""
    model_name = model_name or ollama_config.model_name
    test_prompt = "Respond with exactly: 'Hello, I am working correctly!'"
    
    print(f"🧪 Testing model '{model_name}'...")
    response = call_ollama(test_prompt)
    
    if "❌" not in response:
        print(f"✅ Model '{model_name}' is working!")
        print(f"📝 Response: {response}")
        return True
    else:
        print(f"❌ Model test failed: {response}")
        return False


def call_ollama(prompt: str, system_prompt: str = None) -> str:
    """Call Ollama API with proper error handling"""
    
    if not OLLAMA_AVAILABLE:
        return "❌ Ollama is not available. Please start Ollama and ensure the model is installed."
    
    for attempt in range(ollama_config.max_retries):
        try:
            if system_prompt:
                # Use chat API format
                url = f"{ollama_config.ollama_url}/api/chat"
                payload = {
                    "model": ollama_config.model_name,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": prompt}
                    ],
                    "stream": False,
                    "options": {
                        "temperature": 0.2,
                        "top_p": 0.9,
                        "num_predict": 600
                    }
                }
            else:
                # Use generate API format
                url = f"{ollama_config.ollama_url}/api/generate"
                payload = {
                    "model": ollama_config.model_name,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.2,
                        "top_p": 0.9,
                        "num_predict": 600
                    }
                }
            
            response = requests.post(url, json=payload, timeout=ollama_config.timeout)
            response.raise_for_status()
            
            result = response.json()
            
            if system_prompt:
                return result.get('message', {}).get('content', 'No response').strip()
            else:
                return result.get('response', 'No response').strip()
                
        except requests.exceptions.ConnectionError:
            if attempt == ollama_config.max_retries - 1:
                return f"❌ Could not connect to Ollama at {ollama_config.ollama_url}"
        except requests.exceptions.Timeout:
            if attempt == ollama_config.max_retries - 1:
                return "❌ Request timed out"
        except Exception as e:
            if attempt == ollama_config.max_retries - 1:
                return f"❌ Error: {str(e)}"
    
    return "❌ All retry attempts failed"

# Run initial status check
print("=== Ollama Status Check ===")
# check_ollama_status()

# Uncomment to pull a specific model
# pull_ollama_model("llama3")

# Uncomment to test the current model
# test_ollama_model()

# Check Ollama availability
OLLAMA_AVAILABLE = ollama_config.is_available()
if OLLAMA_AVAILABLE:
    available_models = ollama_config.get_available_models()
    print(f"✅ Ollama is running at {ollama_config.ollama_url}")
    print(f"📋 Available models: {', '.join(available_models) if available_models else 'None'}")
    if ollama_config.model_name in available_models:
        print(f"🎯 Using model: {ollama_config.model_name}")

    else:
        print(f"⚠️ Model '{ollama_config.model_name}' not found. Available: {available_models}")
else:
    print(f"❌ Ollama not available at {ollama_config.ollama_url}")

    print("💡 Start Ollama with: ollama serve")
    print(f"💡 Pull model with: ollama pull {ollama_config.model_name}")


=== Ollama Status Check ===
✅ Ollama is running at http://localhost:11434
📋 Available models: llama3:latest, phi3:latest
🎯 Using model: llama3:latest


In [11]:
import os, re, math, json, time, textwrap, uuid, pathlib
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import requests
from bs4 import BeautifulSoup
import pandas as pd
from pypdf import PdfReader
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer



DATA_DIR = pathlib.Path('rag_iot_data')
DATA_DIR.mkdir(exist_ok=True)

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM = 384  # known dimension for the above model
TOP_K = 5

print('Data dir:', DATA_DIR.resolve())
print('Using embedding model:', MODEL_NAME)

Data dir: /Users/mdjakaria/ncsu-nyu/iot-inspector-client/privacy_agent/rag_iot_data
Using embedding model: sentence-transformers/all-MiniLM-L6-v2


## Starter Device List (edit me!)
These are **example** devices with example URLs (privacy policy and product pages). You can add your own.

**Tip:** If a vendor has a consolidated privacy policy (e.g., Google, Amazon), we still tag chunks with the specific device for easier filtering later.

In [13]:
starter_devices = [
    {
        'device': 'Amazon Echo (Alexa)',
        'brand': 'Amazon',
        'policy_urls': [
            'https://www.amazon.com/gp/help/customer/display.html?nodeId=468496', # Amazon Privacy Notice
            'https://www.amazon.com/Alexa-Privacy-Hub/b?ie=UTF8&node=19149155011', # Alexa Privacy
            'https://www.amazon.com/gp/help/customer/display.html?nodeId=GVP69FUJ48X9DK8V' # Alexa, Echo Devices, and Your Privacy

        ],
        'product_urls': [
            'https://www.amazon.com/echo',
            'https://www.amazon.com/b?node=9818047011',
            'https://www.amazon.com/dp/B0DCCNHWV5?ref=aucc_web_red_xaa_evgn_tx_0002',
            'https://alexa.amazon.com/',
        ],
        'manual_urls': [
            'https://www.amazon.com/gp/help/customer/display.html?nodeId=202016340',
            'https://www.amazon.com/alexa-setup-guide/b?ie=UTF8&node=17978645011'
        ]
    },
    {
        'device': 'Google Nest Thermostat',
        'brand': 'Google',
        'policy_urls': [
            'https://policies.google.com/privacy',
            'https://support.google.com/googlenest/thermostats?hl=en&sjid=4565142753984745171-NA#topic=9361772' # privacy for Nest devices (support)
        ],
        'product_urls': [
            'https://store.google.com/product/nest_thermostat'
        ],
        'manual_urls': [
            'https://support.google.com/googlenest/answer/9248184?hl=en',
            'https://support.google.com/googlenest/answer/9373007?hl=en'
        ]
    },
    {
        'device': 'Ring Video Doorbell',
        'brand': 'Ring',
        'policy_urls': [
            'https://ring.com/privacy'
        ],
        'product_urls': [
            'https://ring.com/video-doorbell-cameras?srsltid=AfmBOorgItpTd1SnsQ6SAk6IjhzhU-IEXDnTV-_6KOUvquqT33CEdyyu',
        ],
        'manual_urls': [
            'https://ring.com/gb/en/support/articles/q39h4/Ring-Device-Manuals?srsltid=AfmBOooIx5dsrxdpFDx_vwEQhEXZZ4XYME5MXk9NSvccuUr9MCnP60Hk',
            'https://ring.com/gb/en/support/articles/t475t/Download-the-Manual-Ring-Video-Doorbell-2nd-Gen?srsltid=AfmBOoqWzmaHcsQ97MoPUTYZNwPpXk1cKTZZRSKrCJUZXXfwBVQl7lIZ'
        ]
    },
    {
        'device': 'Philips Hue',
        'brand': 'Signify',
        'policy_urls': [
            'https://www.signify.com/global/privacy'
        ],
        'product_urls': [
            'https://www.philips-hue.com/en-us/p/hue-white-and-color-ambiance-a19-e26-smart-bulb-60-w-3-pack/046677562786'
        ],
        'manual_urls': [
            'https://www.philips-hue.com/en-us/support/manuals'
        ]
    },
    {
        'device': 'iRobot Roomba',
        'brand': 'iRobot',
        'policy_urls': [
            'https://www.irobot.com/en_US/privacy-policy.html'
        ],
        'product_urls': [
            'https://www.irobot.com/en_US/roomba-max-705-combo-robot-plus-autowash-dock/X185020.html'
        ],
        'manual_urls': [
            'https://homesupport.irobot.com/s/article/843'
        ]
    }
]
pd.DataFrame(starter_devices)

,device,brand,policy_urls,product_urls,manual_urls
0,Amazon Echo (Alexa),Amazon,[https://www.amazon.com/gp/help/customer/displ...,"[https://www.amazon.com/echo, https://www.amaz...",[https://www.amazon.com/gp/help/customer/displ...
1,Google Nest Thermostat,Google,"[https://policies.google.com/privacy, https://...",[https://store.google.com/product/nest_thermos...,[https://support.google.com/googlenest/answer/...
2,Ring Video Doorbell,Ring,[https://ring.com/privacy],[https://ring.com/video-doorbell-cameras?srslt...,[https://ring.com/gb/en/support/articles/q39h4...
3,Philips Hue,Signify,[https://www.signify.com/global/privacy],[https://www.philips-hue.com/en-us/p/hue-white...,[https://www.philips-hue.com/en-us/support/man...
4,iRobot Roomba,iRobot,[https://www.irobot.com/en_US/privacy-policy.h...,[https://www.irobot.com/en_US/roomba-max-705-c...,[https://homesupport.irobot.com/s/article/843]


## Scraping & Parsing Utilities
HTML to clean text, PDF to text, plus a simple chunker. For safety, we set a timeout and basic size limits.

In [14]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (RAG-Research-Demo)'}
MAX_BYTES = 2_000_000  # ~2MB per file

def fetch_url(url: str, timeout: int = 20) -> bytes:
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    if len(r.content) > MAX_BYTES:
        raise ValueError(f'Content too large: {len(r.content)} bytes > {MAX_BYTES}')
    return r.content

def html_to_text(content: bytes) -> str:
    soup = BeautifulSoup(content, 'html.parser')
    # Remove script/style/nav/footer
    for tag in soup(['script','style','nav','footer','header','noscript']):
        tag.decompose()
    texts = [el.get_text(' ', strip=True) for el in soup.find_all(['p','li','h1','h2','h3','h4','h5'])]
    text = '\n'.join([t for t in texts if t])
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def pdf_to_text(content: bytes) -> str:
    from io import BytesIO
    reader = PdfReader(BytesIO(content))
    out = []
    for page in reader.pages:
        t = page.extract_text() or ''
        out.append(t)
    text = '\n'.join(out)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def smart_get_text(url: str) -> str:
    content = fetch_url(url)
    ct = requests.utils.get_encoding_from_headers({'content-type': ''})
    # Heuristic: decide by URL extension or magic bytes
    if url.lower().endswith('.pdf') or (content[:4] == b'%PDF'):
        return pdf_to_text(content)
    return html_to_text(content)

def chunk_text(text: str, max_tokens: int = 350, overlap: int = 40) -> List[str]:
    # Simple whitespace tokenization approximation
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+max_tokens]
        chunks.append(' '.join(chunk))
        i += max(1, max_tokens - overlap)
    return chunks

## Build Corpus (Policies + Product Pages + Manuals)
This will create a dataframe of chunks with metadata, and write it to disk.

In [ ]:
for entry in starter_devices:
    device = entry['device']
    brand = entry.get('brand','')
    for kind, urls in [('privacy_policy', entry.get('policy_urls', [])),
                       ('product_page', entry.get('product_urls', [])),
                       ('manual', entry.get('manual_urls', []))]:
        for url in urls:
            try:
                text = smart_get_text(url)
                if not text or len(text) < 200:
                    print(f"[WARN] Too little text from {url} ({kind})")
                    continue
                chunks = chunk_text(text, max_tokens=350, overlap=40)
                for idx, ch in enumerate(chunks):
                    records.append({
                        'id': str(uuid.uuid4()),
                        'device': device,
                        'brand': brand,
                        'kind': kind,
                        'url': url,
                        'chunk_id': idx,
                        'text': ch
                    })
                print(f"OK: {device} {kind} -> {url} [{len(chunks)} chunks]")
                time.sleep(1)
            except Exception as e:
                print(f"[ERR] {url} -> {e}")

df = pd.DataFrame(records)
print('Total chunks:', len(df))
df_path = DATA_DIR / 'corpus.parquet'
df.to_parquet(df_path, index=False)
df.head()

[ERR] https://www.amazon.com/gp/help/customer/display.html?nodeId=468496 -> 403 Client Error: Forbidden for url: https://www.amazon.com/gp/help/customer/display.html?nodeId=468496
OK: Amazon Echo (Alexa) privacy_policy -> https://www.amazon.com/Alexa-Privacy-Hub/b?ie=UTF8&node=19149155011 [2 chunks]
[ERR] https://www.amazon.com/gp/help/customer/display.html?nodeId=GVP69FUJ48X9DK8V -> 403 Client Error: Forbidden for url: https://www.amazon.com/gp/help/customer/display.html?nodeId=GVP69FUJ48X9DK8V
OK: Amazon Echo (Alexa) product_page -> https://www.amazon.com/echo [2 chunks]
OK: Amazon Echo (Alexa) product_page -> https://www.amazon.com/b?node=9818047011 [2 chunks]
[WARN] Too little text from https://www.amazon.com/dp/B0DCCNHWV5?ref=aucc_web_red_xaa_evgn_tx_0002 (product_page)
OK: Amazon Echo (Alexa) product_page -> https://alexa.amazon.com/ [13 chunks]
[ERR] https://www.amazon.com/gp/help/customer/display.html?nodeId=202016340 -> 403 Client Error: Forbidden for url: https://www.amazon.c

,id,device,brand,kind,url,chunk_id,text
0,5c95b659-486d-48ac-b5bb-37d5160ec486,Amazon Echo (Alexa),Amazon,privacy_policy,https://www.amazon.com/Alexa-Privacy-Hub/b?ie=...,0,Careers Amazon Newsletter About Amazon Accessi...
1,4d9434d3-6b3b-4f12-9188-aa02a3e4b10b,Amazon Echo (Alexa),Amazon,privacy_policy,https://www.amazon.com/Alexa-Privacy-Hub/b?ie=...,1,Woot! Zappos Shoes & Clothing Zappos Ring Smar...
2,eda3db0e-9b79-4dcb-92d4-87bc9745412c,Amazon Echo (Alexa),Amazon,product_page,https://www.amazon.com/echo,0,Careers Amazon Newsletter About Amazon Accessi...
3,11ec5415-d376-4d31-b5e4-0be02a37a00b,Amazon Echo (Alexa),Amazon,product_page,https://www.amazon.com/echo,1,Woot! Zappos Shoes & Clothing Zappos Ring Smar...
4,60e3ec4d-5d22-49b6-acd2-557e7c97e834,Amazon Echo (Alexa),Amazon,product_page,https://www.amazon.com/b?node=9818047011,0,Careers Amazon Newsletter About Amazon Accessi...


## Build Embeddings + FAISS Index
Creates a vector index and saves artifacts to disk for reuse in later sessions.

In [16]:
if len(df) == 0:
    raise RuntimeError('Corpus is empty. Check URLs or add more.')

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(df['text'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

faiss_path = DATA_DIR / 'index.faiss'
faiss.write_index(index, str(faiss_path))

emb_path = DATA_DIR / 'embeddings.npy'
np.save(emb_path, embeddings)

meta_path = DATA_DIR / 'metadata.json'
with open(meta_path, 'w') as f:
    json.dump(df.to_dict(orient='records'), f)

print('Saved index:', faiss_path)
print('Saved embeddings:', emb_path)
print('Saved metadata:', meta_path)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Saved index: rag_iot_data/index.faiss
Saved embeddings: rag_iot_data/embeddings.npy
Saved metadata: rag_iot_data/metadata.json


## Retriever & (Optional) Generator
We implement a basic retriever over FAISS and an optional generator using OpenAI if `OPENAI_API_KEY` is set. If no key is set, you'll still get the top passages with citations.

In [17]:
def load_artifacts():
    df = pd.read_parquet(DATA_DIR / 'corpus.parquet')
    index = faiss.read_index(str(DATA_DIR / 'index.faiss'))
    embeddings = np.load(DATA_DIR / 'embeddings.npy')
    return df, index, embeddings

def retrieve(query: str, k: int = TOP_K) -> pd.DataFrame:
    model = SentenceTransformer(MODEL_NAME)
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q_emb, k)
    hits = df.iloc[I[0]].copy()
    hits['score'] = D[0]
    return hits[['score','device','brand','kind','url','chunk_id','text']]


def synthesize_answer(query: str, hits: pd.DataFrame) -> str:
    context_blocks = []
    for _, row in hits.iterrows():
        cite = f"[{row['device']} | {row['kind']} | {row['url']}]"
        context_blocks.append(f"SOURCE: {cite}\n{row['text']}")
    context_text = '\n\n'.join(context_blocks)

    if OLLAMA_AVAILABLE:
        system_prompt = (
            "You are a careful assistant analyzing IoT device privacy policies and product information. "
            "Answer only using the provided sources. Cite sources inline after each relevant claim using "
            "[device | kind | url] format. If information is missing or unclear, explicitly state so. "
            "Be concise but thorough in your analysis."
        )
        
        user_prompt = f"""Question: {query}

Sources:
{context_text}

Please analyze the sources above to answer the question. Include inline citations and be specific about what each device does regarding the queried topic."""

        try:
            response = call_ollama(user_prompt, system_prompt)
            if response.startswith("❌"):
                return f"{response}\n\n📚 Top passages for reference:\n\n{context_text}"
            return response
        except Exception as e:
            return f"❌ Generation error: {e}\n\n📚 Top passages:\n\n{context_text}"
    else:
        # No Ollama: return the top passages as the 'answer'
        header = f"❌ Ollama not available at {ollama_config.ollama_url}. Showing top retrieved passages with citations.\n\n"
        return header + context_text

# Load artifacts for retrieval
df, index, embeddings = load_artifacts()
print(f'✅ Artifacts loaded. Total chunks: {len(df)}')

✅ Artifacts loaded. Total chunks: 138


## Ask Questions
Try queries like:
- *What data does Ring Video Doorbell collect?*
- *Does Nest Thermostat share data with third parties?*
- *What features does Philips Hue include?*

**Note:** This uses Ollama for local LLM generation. If Ollama is not available, you'll get the retrieved passages with citations instead.

**Troubleshooting:**
- If you see "❌ Ollama not available", run `ollama serve` in terminal
- If model is missing, run `ollama pull llama3` (or your preferred model)
- Use the Ollama management utilities above to check status and pull models

In [23]:
query = "What data does Ring Video Doorbell collect?"
hits = retrieve(query, k=TOP_K)
display(hits)
print('\n---\n')
print(synthesize_answer(query, hits))

,score,device,brand,kind,url,chunk_id,text
67,0.666114,Ring Video Doorbell,Ring,privacy_policy,https://ring.com/privacy,2,forms of communication in an attempt to gain a...
78,0.660276,Ring Video Doorbell,Ring,product_page,https://ring.com/video-doorbell-cameras?srslti...,5,"your Ring Dashboard, you'll be able to view th..."
77,0.640817,Ring Video Doorbell,Ring,product_page,https://ring.com/video-doorbell-cameras?srslti...,4,for up to 180 days. You can manage your video ...
74,0.640467,Ring Video Doorbell,Ring,product_page,https://ring.com/video-doorbell-cameras?srslti...,1,to make neighborhoods safer — starting at the ...
73,0.640456,Ring Video Doorbell,Ring,product_page,https://ring.com/video-doorbell-cameras?srslti...,0,Limited time savings on Certified Refurbished ...



---

According to the provided sources, Ring Video Doorbell collects various types of data, including:

* Videos: Ring records and stores videos when motion is detected by your device (including motion detected via Linked Devices), the video doorbell button is pressed, or you initiate Live View. Stored recordings are automatically deleted following the expiration of your Ring Home Plan's applicable storage period [1].
* Photos: Photos captured will be saved to your Ring account for up to 7 days, so you can review them at any time [2].

Without a subscription to Ring Home, you won't be able to:

* Review videos or images you missed
* Save and share recorded videos
* Use AI to search motion events (Smart Video Search)
* Get more detailed alerts

However, with a Ring Home Plan, you can:

* Save and rewatch the moments you missed for up to 180 days [3]
* Use AI to search your motion events (Smart Video Search) [4]
* Get more detailed alerts
* Share videos and photos with anyone, including

In [24]:
query = "Does Nest Thermostat share data with third parties?"
hits = retrieve(query, k=TOP_K)
display(hits)
print('\n---\n')
print(synthesize_answer(query, hits))

,score,device,brand,kind,url,chunk_id,text
59,0.518001,Google Nest Thermostat,Google,manual,https://support.google.com/googlenest/answer/9...,0,Beginner's guide to the Nest thermostat Learn ...
62,0.514147,Google Nest Thermostat,Google,manual,https://support.google.com/googlenest/answer/9...,3,installation instructions in the Nest app whic...
63,0.489159,Google Nest Thermostat,Google,manual,https://support.google.com/googlenest/answer/9...,4,How Nest thermostats learn Nest thermostat tem...
60,0.465825,Google Nest Thermostat,Google,manual,https://support.google.com/googlenest/answer/9...,1,the touch bar on the right side to control you...
61,0.425830,Google Nest Thermostat,Google,manual,https://support.google.com/googlenest/answer/9...,2,Early-On and how to change settings Thermostat...



---

According to the provided sources, [Google Nest Thermostat | manual | https://support.google.com/googlenest/answer/9248184?hl=en], there is no explicit statement indicating that Nest Thermostat shares data with third parties.

However, it's mentioned in the "How your thermostat works" section that "Your Nest thermostat starts learning on day one, so it’s up to you to teach it good habits to help save energy." [1] This implies that the device learns and adapts to user behavior, which might involve sharing data with Google or other third-party services.

Additionally, the "Save energy with your thermostat" section suggests that users can set Eco Temperatures when no one's home, review their Home Report to monitor energy use, or join an energy-saving program. [2] This implies that some data may be shared with these programs or services.

It's also worth noting that the manual does not explicitly state what kind of data is being shared or with whom. Therefore, it's unclear whether Ne

In [25]:
query = "What features does Philips Hue include?"
hits = retrieve(query, k=TOP_K)
display(hits)
print('\n---\n')
print(synthesize_answer(query, hits))

,score,device,brand,kind,url,chunk_id,text
83,0.578257,Philips Hue,Signify,product_page,https://www.philips-hue.com/en-us/p/hue-white-...,0,Works in every home Dimmable out of the box Ea...
84,0.570351,Philips Hue,Signify,product_page,https://www.philips-hue.com/en-us/p/hue-white-...,1,"rich, powerful light. Soft white The White ran..."
8,0.437601,Amazon Echo (Alexa),Amazon,product_page,https://alexa.amazon.com/,2,"home, simplified – Pair and control devices co..."
85,0.430336,Philips Hue,Signify,product_page,https://www.philips-hue.com/en-us/p/hue-white-...,2,Operational temperature -20°C to 45°C Extra fe...
12,0.422035,Amazon Echo (Alexa),Amazon,product_page,https://alexa.amazon.com/,6,"of privacy controls, including a mic/camera-of..."



---

❌ Request timed out

📚 Top passages for reference:

SOURCE: [Philips Hue | product_page | https://www.philips-hue.com/en-us/p/hue-white-and-color-ambiance-a19-e26-smart-bulb-60-w-3-pack/046677562786]
Works in every home Dimmable out of the box Easy to install yourself A19 - E26 smart bulb - 60 W (3-pack) Philips Hue White and color ambiance E26 bulb 3-pack offers millions of shades of warm-to-cool white and color light to transform your home. These bulbs feature Bluetooth capability for instant light control in one room. Connect to a Hue Bridge to unlock the full suite of smart light features, including the ability to control more than 10 lights — even while away from home. Fitting Light color Shape Model What does this mean? Learn more What does this mean? Learn more Pack Buy 2 or more, save 25% $147.99 End of Season sale Buy now and pay later with Klarna Free Shipping on orders over $49.99 Free Returns for 30 Days 2 Year Warranty Product highlights White and color ambiance Blue

## Add More Devices via CSV (Optional)
Create a CSV with columns: `device,brand,policy_urls,product_urls,manual_urls`.
- URL columns accept **semicolon-separated** values.
- Re-run the build steps to append and re-index.

In [22]:
csv_template = DATA_DIR / 'devices_template.csv'
if not csv_template.exists():
    pd.DataFrame([{ 'device': 'Example Cam', 'brand': 'Acme',
                    'policy_urls': 'https://example.com/privacy',
                    'product_urls': 'https://example.com/product',
                    'manual_urls': 'https://example.com/manual.pdf' }]).to_csv(csv_template, index=False)
csv_template

PosixPath('rag_iot_data/devices_template.csv')

In [21]:
def load_devices_from_csv(path: str) -> List[Dict[str, Any]]:
    df = pd.read_csv(path)
    out = []
    for _, row in df.iterrows():
        def split_urls(s):
            if isinstance(s, str):
                return [u.strip() for u in s.split(';') if u.strip()]
            return []
        out.append({
            'device': row.get('device', ''),
            'brand': row.get('brand', ''),
            'policy_urls': split_urls(row.get('policy_urls', '')),
            'product_urls': split_urls(row.get('product_urls', '')),
            'manual_urls': split_urls(row.get('manual_urls', '')),    
        })
    return out

def append_devices(devices: List[Dict[str, Any]]):
    global df
    recs = []
    for entry in devices:
        device = entry['device']
        brand = entry.get('brand','')
        for kind, urls in [('privacy_policy', entry.get('policy_urls', [])),
                           ('product_page', entry.get('product_urls', [])),
                           ('manual', entry.get('manual_urls', []))]:
            for url in urls:
                try:
                    text = smart_get_text(url)
                    if not text or len(text) < 200:
                        print(f"[WARN] Too little text from {url} ({kind})")
                        continue
                    chunks = chunk_text(text, max_tokens=350, overlap=40)
                    for idx, ch in enumerate(chunks):
                        recs.append({
                            'id': str(uuid.uuid4()),
                            'device': device,
                            'brand': brand,
                            'kind': kind,
                            'url': url,
                            'chunk_id': idx,
                            'text': ch
                        })
                    print(f"OK: {device} {kind} -> {url} [{len(chunks)} chunks]")
                    time.sleep(1)
                except Exception as e:
                    print(f"[ERR] {url} -> {e}")
    if recs:
        new_df = pd.DataFrame(recs)
        df = pd.concat([df, new_df], ignore_index=True)
        df.to_parquet(DATA_DIR / 'corpus.parquet', index=False)
        # rebuild embeddings + index
        model = SentenceTransformer(MODEL_NAME)
        new_emb = model.encode(df['text'].tolist(), batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
        idx = faiss.IndexFlatIP(EMBED_DIM)
        idx.add(new_emb)
        faiss.write_index(idx, str(DATA_DIR / 'index.faiss'))
        np.save(DATA_DIR / 'embeddings.npy', new_emb)
        print('Index updated with new devices.')
    else:
        print('No valid records added.')

## Export Q&A Pairs (Optional)
You can export top-k contexts for your own fine-tuning or testing.

In [ ]:
def export_contexts(query: str, path: str):
    hits = retrieve(query, k=TOP_K)
    payload = {
        'query': query,
        'contexts': hits.to_dict(orient='records'),
        'timestamp': datetime.utcnow().isoformat()+'Z'
    }
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2)
    return path

out_path = DATA_DIR / 'sample_context.json'
export_contexts('Does Nest Thermostat share data with third parties?', str(out_path))
out_path

## IoT Device Analysis from FingerBank Data
### TODO: Update device list from other sources: https://github.com/jakariamd/IoT-Measurement/tree/main/Device%20Identification
Analyze the finger_banks_manual_inspection.csv to extract unique vendor-device pairs for privacy policy analysis.

In [27]:
import pandas as pd
from collections import defaultdict
import os
import json

# Read the finger_banks_manual_inspection.csv file
csv_file_path = DATA_DIR / 'device_database' / 'finger_banks_manual_inspection.csv'

if os.path.exists(csv_file_path):
    # Read the CSV file
    fingerbank_df = pd.read_csv(csv_file_path)
    
    print(f"📊 Loaded {len(fingerbank_df)} records from finger_banks_manual_inspection.csv")
    print(f"🔍 Columns: {list(fingerbank_df.columns)}")
    
    # Extract unique vendor-device pairs
    vendor_device_dict = {}
    
    for _, row in fingerbank_df.iterrows():
        vendor_name = row.get('vendor_name', '').strip()
        fingerbank_label = row.get('fingerbank_label', '').strip()
        
        # Skip empty values
        if not vendor_name or not fingerbank_label:
            continue
        
        # Split fingerbank_label by '/' and keep the last part (device name)
        device_name = fingerbank_label.split('/')[-1].strip()
        
        # Create vendor-device pair
        if vendor_name not in vendor_device_dict:
            vendor_device_dict[vendor_name] = set()
        
        vendor_device_dict[vendor_name].add(device_name)
    
    # Convert sets to lists for easier handling
    vendor_device_final = {vendor: list(devices) for vendor, devices in vendor_device_dict.items()}
    
    # Display results
    print(f"\n📋 Found {len(vendor_device_final)} unique vendors:")
    print("=" * 50)
    
    total_devices = 0
    for vendor, devices in sorted(vendor_device_final.items()):
        total_devices += len(devices)
        print(f"\n🏢 {vendor} ({len(devices)} devices):")
        for device in sorted(devices):
            print(f"   • {device}")
    
    print(f"\n📈 Summary:")
    print(f"   • Total vendors: {len(vendor_device_final)}")
    print(f"   • Total unique devices: {total_devices}")
    print(f"   • Average devices per vendor: {total_devices/len(vendor_device_final):.1f}")
    
    # Show some statistics
    print(f"\n🔝 Top vendors by device count:")
    vendor_counts = [(vendor, len(devices)) for vendor, devices in vendor_device_final.items()]
    vendor_counts.sort(key=lambda x: x[1], reverse=True)
    
    for vendor, count in vendor_counts[:10]:  # Top 10
        print(f"   • {vendor}: {count} devices")
    
    # Store the result for later use
    print(f"\n💾 Data stored in 'vendor_device_final' dictionary")
    
    # Save to files
    print(f"\n💾 Saving data to files...")
    
    # Save as JSON
    json_file_path = DATA_DIR / 'device_database' / 'vendor_device_mapping.json'
    with open(json_file_path, 'w', encoding='utf-8') as f:
        json.dump(vendor_device_final, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved JSON: {json_file_path}")
    
    # Save as CSV (flattened format)
    csv_output_path = DATA_DIR / 'device_database' /  'vendor_device_mapping.csv'
    flattened_data = []
    for vendor, devices in vendor_device_final.items():
        for device in devices:
            flattened_data.append({
                'vendor_name': vendor,
                'device_name': device
            })
    
    vendor_device_df = pd.DataFrame(flattened_data)
    vendor_device_df.to_csv(csv_output_path, index=False)
    print(f"✅ Saved CSV: {csv_output_path}")
    
    # Save summary statistics
    summary_file_path = DATA_DIR / 'device_database' / 'vendor_device_summary.json'
    summary_data = {
        'total_vendors': len(vendor_device_final),
        'total_unique_devices': total_devices,
        'average_devices_per_vendor': round(total_devices/len(vendor_device_final), 2),
        'top_vendors': [{'vendor': vendor, 'device_count': count} for vendor, count in vendor_counts[:10]],
        'generated_timestamp': pd.Timestamp.now().isoformat()
    }
    
    with open(summary_file_path, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved summary: {summary_file_path}")
    
    print(f"\n📂 Files saved:")
    print(f"   • JSON format: {json_file_path.name}")
    print(f"   • CSV format: {csv_output_path.name}")
    print(f"   • Summary stats: {summary_file_path.name}")
    
else:
    print(f"❌ File not found: {csv_file_path}")
    print("🔍 Available files in rag_iot_data:")
    if DATA_DIR.exists():
        for file in DATA_DIR.iterdir():
            print(f"   • {file.name}")
    vendor_device_final = {}

📊 Loaded 18995 records from finger_banks_manual_inspection.csv
🔍 Columns: ['user_key', 'device_id', 'suspected_pc', 'user_label', 'user_agent_info', 'oui_friendly', 'oui_raw', 'netdisco_info', 'user_country', 'timezone', 'fingerbank_label', 'device_category', 'vendor_name', 'inspected']

📋 Found 127 unique vendors:

🏢 ADB (1 devices):
   • ADB Set Top Box

🏢 AT&T (1 devices):
   • Google Chromecast

🏢 Amazon (9 devices):
   • Amazon Alexa
   • Amazon Echo
   • Amazon Echo Dot
   • Amazon Echo Show
   • Amazon Fire TV
   • Amazon Fire TV 3
   • Amazon Fire TV Edition
   • Amazon Fire TV Stick
   • Amazon Fire TV Stick 4K

🏢 Amcrest (1 devices):
   • Amcrest Security Camera

🏢 Apple (3 devices):
   • Apple TV
   • Apple TV 4K
   • Apple TV HD 4

🏢 Arcadyan (1 devices):
   • Arcadyan HMB2260

🏢 Arlo (1 devices):
   • Arlo  Q Plus

🏢 Axis (1 devices):
   • Axis Communications

🏢 Baggen (1 devices):
   • Frontier Silicon

🏢 Belkin (2 devices):
   • Belkin WeMo
   • Belkin Wireless Router

🏢